# Pivot Tables

## Introduction

A pivot table reshapes a DataFrame so that unique values of one column become the column headers, making cross-group comparisons much easier to read. This notebook builds from groupby aggregation through multi-hierarchical indexing to pivot tables and the `stack`/`unstack` operations that move between them.

## Objectives

You will be able to:

- Distinguish between wide and long data formats
- Interpret and navigate a multi-hierarchical (MultiIndex) DataFrame
- Use `.pivot()` to reshape a grouped DataFrame into a pivot table
- Use `.stack()` and `.unstack()` to move between index levels and columns

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

df = pd.read_csv('data/pivot_tables_pandas_lab/causes_of_death.tsv', delimiter='\t')
print(df.shape)
df.head()

---

## Wide vs Long Format

**Wide format** — each variable is its own column, each row is one observation. The format you're used to seeing.

**Long format** — each row is one measurement for one observation at one point in time. Repeated observations for the same subject span multiple rows.

![wide vs long](assets/pivot_tables_pandas/Image_200_wide_v_long.png)

Long format enables multi-hierarchical indexing, which makes aggregating over combinations of variables very clean. The CDC dataset above is already in long format — a given state appears across many rows, one per age group / gender / race combination.

---

## Data Preparation

The `Population` column is stored as a string because some cells contain `'Not Applicable'`. Drop those rows and cast to integer before aggregating.

In [ ]:
print(df['Population'].value_counts().head())
print(f"\n'Not Applicable' rows: {(df['Population'] == 'Not Applicable').sum()}")

In [ ]:
na_rows = df[df['Population'] == 'Not Applicable'].index
df.drop(na_rows, inplace=True)
df['Population'] = df['Population'].astype('int64')
print(f"Rows remaining: {len(df)}  |  Population dtype: {df['Population'].dtype}")

---

## Groupby Aggregation

Aggregate first to reduce the data to the level you want to pivot on.

In [ ]:
# Total deaths by state
df.groupby('State')['Deaths'].sum().sort_values(ascending=False).head(8)

In [ ]:
# Bar chart — total deaths by state
df.groupby('State')['Deaths'].sum().sort_values().plot(
    kind='barh', figsize=(10, 14), title='Total Deaths by State'
)
plt.xlabel('Deaths')
plt.tight_layout()
plt.show()

In [ ]:
# Multi-stat aggregation — mean, min, max, std for Deaths and Population by State + Gender
grouped = df.groupby(['State', 'Gender'])['Deaths', 'Population'].agg(['mean', 'min', 'max', 'std'])
grouped.head()

---

## Multi-Hierarchical Indexing

When you group by multiple columns, pandas creates a **MultiIndex** — multiple levels of row labels. The example above groups by `State` and `Gender`, producing a two-level index:

![multi-hierarchical index — State and Gender](assets/pivot_tables_pandas/pt1.png)

Three levels looks like this:

![multi-hierarchical index — State, Gender, Race](assets/pivot_tables_pandas/pt2.png)

In [ ]:
# Inspect the multi-level index
print(grouped.index[:6])

In [ ]:
# Reset to a flat integer index — State and Gender become regular columns
grouped = grouped.reset_index()

# The columns are now also multi-level — flatten them
cols0 = grouped.columns.get_level_values(0)
cols1 = grouped.columns.get_level_values(1)
grouped.columns = [
    f"{c0}_{c1}" if c1 else c0
    for c0, c1 in zip(cols0, cols1)
]

grouped.head()

---

## Pivot Tables

A pivot table rotates a column of values into column headers, making cross-group comparisons immediate. In pandas:

```python
df.pivot(index='row_label', columns='col_label', values='value')
```

![pivot table concept](assets/pivot_tables_pandas/pt3.png)

In [ ]:
# Pivot: State as rows, Gender as columns, mean deaths as values
pivot = grouped.pivot(index='State', columns='Gender', values='Deaths_mean')
pivot.head()

In [ ]:
# Grouped bar chart from the pivot table
pivot.plot(kind='barh', figsize=(10, 14), title='Mean Deaths by State and Gender')
plt.xlabel('Mean deaths')
plt.tight_layout()
plt.show()

In [ ]:
# Stacked version — easier to compare totals
pivot.plot(kind='barh', figsize=(10, 14), stacked=True, title='Mean Deaths by State and Gender (stacked)')
plt.xlabel('Mean deaths')
plt.tight_layout()
plt.show()

---

## Stack and Unstack

`.stack()` and `.unstack()` move data between the index and the columns — they are the programmatic equivalent of pivoting.

![unstack diagram](assets/pivot_tables_pandas/Image_201_unstack.png)

- **`.unstack()`** — takes the innermost index level and spreads it across columns (wide)
- **`.stack()`** — takes the innermost column level and folds it into the index (long)

In [ ]:
# Unstack the pivot — Gender moves from columns back into the index
unstacked = pivot.unstack()
print(type(unstacked))
unstacked.head(8)

In [ ]:
# Unstack again — now completely flat (Gender × State combinations as index)
unstacked.unstack().head()

In [ ]:
# Stack the pivot — folds the Gender column back into the index
pivot.stack().head(8)

---

## Summary

In this notebook you learned how to:

- Distinguish wide and long data formats and understand when each is useful
- Interpret a MultiIndex and flatten it with `.reset_index()` and column renaming
- Use `.pivot()` to reshape a grouped DataFrame into a pivot table for visualisation
- Use `.stack()` and `.unstack()` to move data between index levels and columns

Next: [06 — Data Cleaning Project](06_data_cleaning_project.ipynb)